# Voice AI Platforms

**Module:** 16 — Speech AI

CPaaS, contact-center AI, cloud speech suites, and open stacks — plus compliance.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Categorize voice platforms and when to use each
- Sketch integrations with telephony webhooks
- Apply a compliance checklist (consent, recording, PCI, biometrics)
- Compare build vs buy for voice agents


## Platform Categories

### Definition
Voice platforms package telephony, ASR/TTS, orchestration, and sometimes LLM agents as managed services.

### Why it matters
Rebuilding SIP trunks and call queues rarely differentiates your product.

### How it works
Categories: CPaaS (Twilio-like), CCaaS + AI, cloud speech suites, open-source stacks, device SDKs.

### Intuition
Choose a chassis, then invent the drive experience.

### Pitfalls
- Buying five overlapping platforms
- Ignoring recording laws by region

### When to use
Enterprise and product voice roadmaps.


### Categories

| Category | Examples (illustrative) | Best for |
|----------|-------------------------|----------|
| CPaaS | Twilio, Vonage, Telnyx | Programmable calls/SMS |
| CCaaS + AI | Genesys, Amazon Connect… | Contact centers |
| Cloud speech | Azure/Google/AWS speech | STT/TTS building blocks |
| Realtime LLM voice | OpenAI Realtime etc. | Duplex agents |
| Open stack | Asterisk + Whisper + Piper | Control / cost at scale |

```mermaid
flowchart LR
  PSTN[PSTN / SIP] --> CP[CPaaS / CCaaS]
  CP --> Bot[Voice agent runtime]
  Bot --> ASR
  Bot --> LLM
  Bot --> TTS
  Bot --> CRM[CRM / tools]
```


In [ ]:
# Demo 1: inbound call webhook sketch
import json

def handle_inbound(event: dict) -> dict:
    # Twilio-like conceptual response
    return {
        "actions": [
            {"say": "Thanks for calling Acme."},
            {"start_agent": {"workflow": "support_v2", "from": event.get("from")}},
        ]
    }

print(json.dumps(handle_inbound({"from": "+15551234567", "to": "+15557654321"}), indent=2))


In [ ]:
# Demo 2: build vs buy scorecard
def build_vs_buy(answers: dict) -> str:
    score_buy = 0
    if answers.get("need_pstn_fast"): score_buy += 2
    if answers.get("compliance_pack_needed"): score_buy += 2
    if answers.get("small_team"): score_buy += 1
    if answers.get("unique_ondevice_req"): score_buy -= 2
    if answers.get("extreme_volume_cost_sensitivity"): score_buy -= 1
    return "buy/platform" if score_buy >= 2 else "build/hybrid"

print(build_vs_buy({"need_pstn_fast": True, "small_team": True, "compliance_pack_needed": True}))
print(build_vs_buy({"unique_ondevice_req": True, "extreme_volume_cost_sensitivity": True}))


## Integration Sketch

### Definition
Integrations connect carriers/webhooks to agent runtimes, CRMs, and storage with identity and retry semantics.

### Why it matters
Most outages are integration bugs, not model quality.

### How it works
Inbound webhook → auth → session create → media stream URL → agent → post-call summary to CRM; idempotent event keys.

### Intuition
The switchboard must know the caller before the genius answers.

### Pitfalls
- No idempotency on call-completed events
- Storing recordings in product DB forever

### When to use
Any telephony-connected agent.


In [ ]:
# Demo 3: idempotent call event processor
SEEN = set()

def on_event(event_id: str, payload: dict):
    if event_id in SEEN:
        return {"status": "duplicate"}
    SEEN.add(event_id)
    # persist summary…
    return {"status": "processed", "call": payload.get("call_id")}

print(on_event("evt_1", {"call_id": "c1"}))
print(on_event("evt_1", {"call_id": "c1"}))


In [ ]:
# Demo 4: post-call CRM note
def crm_note(transcript: str, slots: dict) -> dict:
    return {
        "subject": f"Voice: {slots.get('intent', 'general')}",
        "body": transcript[:500],
        "fields": slots,
        "api_key": "YOUR_CRM_API_KEY",
    }
print({k: v for k, v in crm_note("Canceled order 4455", {"intent": "cancel_order", "order_id": "4455"}).items() if k != "api_key"})


### Compliance checklist

| Topic | Questions |
|-------|-----------|
| Consent | One-party/two-party recording laws? |
| PCI | Are card numbers spoken? Pause recording? |
| Retention | Audio TTL vs transcript TTL? |
| Biometrics | Voiceprint features legally reviewed? |
| Residency | Where do vendors process media? |
| Disclosure | Synthetic voice disclosed if required? |
| Access | Who can listen to recordings? |


In [ ]:
# Demo 5: recording policy gate
def recording_allowed(region: str, consent: bool, pci_flow: bool) -> str:
    if not consent:
        return "deny_record"
    if pci_flow:
        return "record_with_pause_on_capture"
    if region in {"EU", "IN"}:
        return "record_with_strict_ttl"
    return "record_ok"
print(recording_allowed("US", True, True))
print(recording_allowed("EU", False, False))


In [ ]:
# Demo 6: platform capability matrix query
MATRIX = {
    "cpaas": {"pstn": True, "streams": True, "llm_agent": False},
    "realtime_llm": {"pstn": False, "streams": True, "llm_agent": True},
    "ccaas": {"pstn": True, "streams": True, "llm_agent": True},
}
def needs(pstn, llm):
    return [name for name, c in MATRIX.items() if (not pstn or c["pstn"]) and (not llm or c["llm_agent"])]
print("pstn+llm", needs(True, True))
print("llm only", needs(False, True))


### Checklist — Platform compliance

- [ ] Recording consent flow verified
- [ ] PCI pause/resume designed
- [ ] Vendor DPA / residency reviewed
- [ ] Idempotent webhooks
- [ ] Access control for recordings


### Try it yourself — Platforms

1. Map your product to a category + one backup.
2. Write a call-completed webhook handler with signature verify stub.
3. Draft retention TTLs by data type.

**Stretch:** Price 100k minutes on two CPaaS calculators.


### Try it yourself — Compliance

1. List regions you operate in and recording rules (research).
2. Design a PCI pause state in the turn machine.


## Knowledge Check

**Q1.** Why idempotent telephony webhooks?

<details><summary>Answer</summary>

Carriers retry; double CRM updates / double charges are common incidents.

</details>

**Q2.** What is PCI pause?

<details><summary>Answer</summary>

Stop recording/transcript while collecting card data to reduce compliance scope.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `CPaaS` | Communications platform as a service |
| `CCaaS` | Contact center as a service |
| `SIP` | Session initiation protocol |
| `PSTN` | Public switched telephone network |
| `DPA` | Data processing agreement |
| `PCI` | Payment card industry standards |


## Key Takeaways

- Pick platform category from PSTN/LLM/compliance needs
- Integrations need idempotency and identity
- Recording/PCI/biometric rules are product features
- Build vs buy is a scorecard, not a slogan


## Production Incident Patterns — voice platforms

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Users talk over bot | No barge-in / bad VAD | Tune endpointing; cancel TTS |
| High WER in field | Noise/codec mismatch | Denoise; match sample rate |
| Creepy voice clone | Weak consent policy | Watermark + allow-list |
| 800ms+ dead air | Cascaded STT→LLM→TTS | Speculative TTS; S2S; stream |
| Compliance scare | Raw audio retention | TTL + transcript-only default |

```
Voice control loop:
  mic -> VAD -> ASR partials -> NLU/LLM -> TTS stream -> speaker
                ^                | tools/HITL
                +-- transcripts/metrics/audit --+
```


In [ ]:
# Cross-cutting: never log raw secrets or full audio bytes
import hashlib, json

def audio_audit(user_id: str, wav_bytes: bytes, meta: dict) -> dict:
    return {
        "user_id": user_id,
        "sha256_16": hashlib.sha256(wav_bytes).hexdigest()[:16],
        "nbytes": len(wav_bytes),
        "meta": {k: v for k, v in meta.items() if k not in {"api_key", "authorization"}},
        "topic": "voice platforms",
    }

print(json.dumps(audio_audit("u1", b"RIFF....", {"model": "whisper", "api_key": "YOUR_OPENAI_API_KEY"})))


## Mini Case Study — voice platforms

**Scenario:** A support org replaces IVR menus with a voice agent. Pilot NPS soars.
**Month 2:** Accents under-served; callers interrupted mid-sentence; recordings retained 2 years.

**Retro questions**
1. What was the latency budget (ASR+LLM+TTS)?
2. Was barge-in tested with noisy headsets?
3. Retention: audio vs transcript vs redacted entities?
4. Which intents require human transfer?

**Design rule:** conversational voice is a real-time distributed system — optimize the path, not only model quality.


In [ ]:
# Cross-cutting: latency budget checker
from dataclasses import dataclass

@dataclass
class VoiceBudget:
    asr_ms: int = 300
    llm_first_token_ms: int = 400
    tts_first_audio_ms: int = 200
    network_ms: int = 100
    def total(self): return self.asr_ms + self.llm_first_token_ms + self.tts_first_audio_ms + self.network_ms
    def ok(self, sla=900): return self.total() <= sla

b = VoiceBudget()
print("voice platforms", "total_ms", b.total(), "ok", b.ok())
print("tight", VoiceBudget(500, 600, 300, 150).ok())


### Try it yourself — voice platforms ops

1. Draft an on-call runbook bullet list for voice platforms when p95 turn latency > SLA.
2. Sketch metrics: WER proxy, barge-in rate, transfer rate, audio retention age.
